# Inference: one unlabeled test sequence
Run on **Godfather (Python 3.11 | Torch + Paddle GPU)**. No labels, training data,
GT matching, or accuracy calculations are required. Change `SEQUENCE_ID` to try
another test sequence, then run top to bottom.

Original Magi detections/associations → selected Paddle/Magi OCR → fine-tuned
character crop embeddings → duplicate cleanup → cross-page entities → dialogue JSON.
Original text-box order stays fixed. `E1`, `E2`, ... identify characters only within
this sequence; they are inferred IDs, not character names.

Use the two-block checkpoint and its previously selected **clustering** threshold
`0.82`, from `fineTuningMagiCharSimHeads2/testMagiCropEncoder.ipynb` (selected epoch 20).
No thresholds are selected from these unlabeled test pages. Duplicate-cleanup
thresholds remain the existing heuristics: IoU `0.90`, cosine `0.95`.

The viewer below lets you choose a page, OCR source, and visible annotations.
Inference caches are isolated under `cache_inference/test/<sequence_id>/`.


## 1. Imports and settings


In [1]:
import torch  # Must precede Paddle on this Windows installation.
import paddle
import gc
import copy
import hashlib
import json
import sys
import unicodedata
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from transformers import AutoModel
from paddleocr import PaddleOCR

print('Python:', sys.executable)
print('Torch:', torch.__version__, '| Paddle:', paddle.__version__)


C:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
W1013 18:49:49.085000 13176 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Python: C:\Users\ESHAAN\BYTE\godfather.venv\Scripts\python.exe
Torch: 2.14.0+cu126 | Paddle: 3.2.0


In [2]:
MAGI_ROOT = Path(r'C:\Users\ESHAAN\BYTE\Applied-Task\Magi')
MODEL_PATH = Path(r'C:\MODELS\magiv2')
SPLIT = 'test'
SEQUENCE_ID = 'seq_14469aeb38963085'
LANGUAGE = 'en'
PREFERRED_OCR = 'paddle'
CHECKPOINT_PATH = MAGI_ROOT /"FINETUNING-EXPERMENTS"/ 'fineTuningMagiCharSimHeads2' / 'magi_crop_last_2_blocks.pt'
ENTITY_THRESHOLD = 0.82  # Clustering threshold selected on labeled validation data before inference.
THRESHOLD_CHECKPOINT_EPOCH = 20
CHARACTER_CLEANUP_SETTINGS = dict(min_iou=0.90, min_cosine=0.95,
                                  algorithm='contained_direct_outer_match_v1')
# Bump after replacing model weights without changing their file metadata.
MODEL_REVISION = 'local-magiv2-1'
FORCE = dict(magi_raw=False, embeddings=False, paddle_ocr=False,
             ocr_selection=False, character_cleanup=False, entities=False, text_entities=False)
# Bump the affected version when editing a stage's algorithm.
STAGE_VERSIONS = dict(magi_raw=1, embeddings=1, paddle_ocr=1,
                     ocr_selection=1, character_cleanup=1, entities=1, text_entities=1)
PADDLE_SETTINGS = dict(lang=LANGUAGE, device='gpu:0', ocr_version='PP-OCRv5',
                      use_doc_orientation_classify=False, use_doc_unwarping=False,
                      use_textline_orientation=False)
SELECTOR_SETTINGS = dict(prefer=PREFERRED_OCR, check='unicode_text_validity_v1')


In [3]:
SEQUENCE_ROOT = MAGI_ROOT / 'TaskData' / SPLIT / 'images' / SEQUENCE_ID
CACHE_DIR = MAGI_ROOT / 'paddleOCR-Magi' / 'cache_inference' / SPLIT / SEQUENCE_ID
assert SEQUENCE_ROOT.is_dir(), f'Missing sequence: {SEQUENCE_ROOT}'
assert MODEL_PATH.is_dir(), f'Missing original model: {MODEL_PATH}'
assert CHECKPOINT_PATH.is_file(), f'Missing two-block checkpoint: {CHECKPOINT_PATH}'
PAGE_FILES = sorted(SEQUENCE_ROOT.glob('*.png'))
assert PAGE_FILES, 'No PNG pages found'
chapter_pages, crop_pages = [], []
for path in PAGE_FILES:
    with Image.open(path) as image:
        chapter_pages.append(np.array(image.convert('L').convert('RGB')))
        crop_pages.append(np.array(image.convert('RGB')))  # Match the fine-tuning crop input path.
print('Sequence:', SEQUENCE_ID, '| pages:', [p.name for p in PAGE_FILES])
print('Cache:', CACHE_DIR)


Sequence: seq_14469aeb38963085 | pages: ['01.png', '02.png', '03.png']
Cache: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_inference\test\seq_14469aeb38963085


## 2. Cache and checkpoint checks
Checkpoint SHA256 enters the embedding cache signature. A different checkpoint
recomputes embeddings and dependent stages; changing the entity threshold only
recomputes entities and their outputs. Original Magi predictions and OCR have
separate cache signatures. No inference cache depends on labels.


In [4]:
def json_ready(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    return value


def digest(data):
    text = json.dumps(json_ready(data), sort_keys=True, ensure_ascii=True, allow_nan=False)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def file_hash(path):
    with path.open('rb') as file:
        return hashlib.file_digest(file, 'sha256').hexdigest()


def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(json_ready(data), ensure_ascii=True,
                                    indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def model_fingerprint():
    files = []
    for path in sorted(MODEL_PATH.rglob('*')):
        if path.is_file() and path.suffix in {'.json', '.py', '.safetensors', '.bin', '.pt', '.pth'}:
            stat = path.stat()
            files.append(dict(name=str(path.relative_to(MODEL_PATH)), size=stat.st_size,
                              mtime_ns=stat.st_mtime_ns,
                              sha256=file_hash(path) if path.suffix in {'.json', '.py'} else None))
    return dict(path=str(MODEL_PATH), revision=MODEL_REVISION, files=files)


PACKAGES = {name: version(name) for name in ['torch', 'torchvision', 'transformers',
            'paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy', 'pillow']}
PAGE_METADATA = [dict(name=p.name, sha256=file_hash(p), width=image.shape[1], height=image.shape[0])
                 for p, image in zip(PAGE_FILES, chapter_pages)]
INPUTS = dict(sequence_id=SEQUENCE_ID, split=SPLIT, pages=PAGE_METADATA,
              preprocessing='grayscale_then_RGB')
MODEL_INFO = model_fingerprint()
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
assert checkpoint['version'] == 2 and checkpoint['settings']['trained_blocks'] == 2
assert checkpoint['best_epoch'] == THRESHOLD_CHECKPOINT_EPOCH, (
    'Checkpoint epoch changed. Set ENTITY_THRESHOLD and THRESHOLD_CHECKPOINT_EPOCH from its validation clustering run.')
for name, expected in checkpoint['base_model_hashes'].items():
    assert file_hash(MODEL_PATH / name) == expected, f'Original model changed: {name}'
CHECKPOINT_INFO = dict(path=str(CHECKPOINT_PATH), sha256=file_hash(CHECKPOINT_PATH),
                       epoch=checkpoint['best_epoch'], trained_blocks=2)
manifest = dict(schema_version=1, inputs=INPUTS, model=MODEL_INFO, packages=PACKAGES,
                crop_encoder=CHECKPOINT_INFO, stages={})
print('Crop checkpoint epoch:', checkpoint['best_epoch'], '| clustering threshold:', ENTITY_THRESHOLD)
MANIFEST_PATH = CACHE_DIR / 'manifest.json'
if MANIFEST_PATH.exists():
    try:
        old_manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        if old_manifest.get('inputs') == INPUTS:
            manifest['stages'] = old_manifest.get('stages', {})
    except (ValueError, OSError):
        pass


def stage_ref(name):
    entry = manifest['stages'][name]
    return dict(signature=entry['signature'], data_hash=entry['data_hash'])


def cached_stage(name, settings, compute):
    signature = digest(dict(inputs=INPUTS, algorithm_version=STAGE_VERSIONS[name], settings=settings))
    path = CACHE_DIR / f'{name}.json'
    payload = None
    if path.exists() and not FORCE[name]:
        try:
            candidate = json.loads(path.read_text(encoding='utf-8'))
            if candidate['signature'] == signature and digest(candidate['data']) == candidate['data_hash']:
                payload = candidate
        except (ValueError, KeyError, OSError):
            pass
    if payload is None:
        print(f'RUN   {name}')
        data = json_ready(compute())
        payload = dict(signature=signature, data_hash=digest(data),
                       created_at=datetime.now(timezone.utc).isoformat(), data=data)
        write_json(path, payload)
    else:
        print(f'CACHE {name}')
    manifest['stages'][name] = dict(file=path.name, signature=signature,
                                  data_hash=payload['data_hash'], settings=settings,
                                  algorithm_version=STAGE_VERSIONS[name], created_at=payload['created_at'])
    write_json(MANIFEST_PATH, manifest)
    return payload['data']


Crop checkpoint epoch: 20 | clustering threshold: 0.82


## 3. Original Magi predictions, then fine-tuned crop embeddings
Magi creates detections, page-local clusters, text–character links, OCR, and
essential-text flags using original weights. Only after that prediction finishes
are the final two crop-ViT blocks replaced. Thus the checkpoint only changes the
explicit character embeddings used by cleanup and cross-page matching.
The final LayerNorm and all remaining weights retain their original values.


In [5]:
magi_model = None


def get_magi():
    global magi_model
    if magi_model is None:
        assert torch.cuda.is_available(), 'Torch CUDA unavailable'
        # Normal tensors allow the later checkpoint copy; only predictions use inference mode.
        with torch.inference_mode(False):
            magi_model = AutoModel.from_pretrained(str(MODEL_PATH), trust_remote_code=True,
                                                  local_files_only=True).cuda().eval()
    return magi_model


def predict_magi():
    model = get_magi()
    with torch.inference_mode():
        return model.do_chapter_wide_prediction(
            chapter_pages, {'images': [], 'names': []}, use_tqdm=True, do_ocr=True)


def predict_embeddings():
    boxes = [result['characters'] for result in magi_raw_results]
    if not any(boxes):
        return [[] for _ in boxes]
    model = get_magi()
    last = len(model.crop_embedding_model.encoder.layer) - 1
    assert last >= 1
    prefixes = (f'crop_embedding_model.encoder.layer.{last-1}.',
                f'crop_embedding_model.encoder.layer.{last}.')
    parameters = dict(model.named_parameters())
    expected_names = {name for name in parameters if name.startswith(prefixes)}
    assert tuple(checkpoint['trainable_prefixes']) == prefixes
    assert set(checkpoint['state_dict']) == expected_names
    # Patch only the two crop blocks after original Magi prediction is complete.
    with torch.no_grad():
        for name, tensor in checkpoint['state_dict'].items():
            assert tensor.shape == parameters[name].shape
            parameters[name].copy_(tensor.to(parameters[name].device))
    model.eval()
    with torch.inference_mode():
        embeddings = model.predict_crop_embeddings(crop_pages, boxes, mask_ratio=0.0, batch_size=32)
        return [torch.nn.functional.normalize(x, p=2, dim=1) if len(x) else x for x in embeddings]


def validate_magi(results):
    assert len(results) == len(chapter_pages), 'Magi page count mismatch'
    for result in results:
        assert len(result['texts']) == len(result['ocr']), 'Text/OCR count mismatch'
        assert len(result['characters']) == len(result['character_cluster_labels']), 'Character/cluster count mismatch'
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']), 'Invalid text association index'
            assert 0 <= char_index < len(result['characters']), 'Invalid character association index'


In [6]:
try:
    magi_raw_results = cached_stage('magi_raw',
        dict(model=MODEL_INFO, torch=PACKAGES['torch'], torchvision=PACKAGES['torchvision'],
             transformers=PACKAGES['transformers'], do_ocr=True), predict_magi)
    validate_magi(magi_raw_results)
    embeddings_per_page = cached_stage('embeddings',
        dict(magi=stage_ref('magi_raw'), model=MODEL_INFO, checkpoint=CHECKPOINT_INFO,
             crop_preprocessing='original_RGB', mask_ratio=0.0, batch_size=32, normalization='L2',
             torch=PACKAGES['torch'], transformers=PACKAGES['transformers']), predict_embeddings)
finally:
    magi_model = None
    gc.collect()
    torch.cuda.empty_cache()

assert len(embeddings_per_page) == len(magi_raw_results)
for result, embeddings in zip(magi_raw_results, embeddings_per_page):
    assert len(embeddings) == len(result['characters']), 'Character/embedding count mismatch'
    assert np.isfinite(np.asarray(embeddings)).all(), 'Nonfinite embeddings'
per_page_results = copy.deepcopy(magi_raw_results)
print('Text counts:', [len(r['texts']) for r in per_page_results])


CACHE magi_raw
CACHE embeddings
Text counts: [15, 14, 12]


## 4. Paddle OCR on original text boxes

<details>
<summary>How this stage works</summary>

Runs Paddle directly on GPU. Records retain page order and original text-box order. Coordinates are rounded outward and clipped only for cropping; original boxes are retained.

Invalid crops and per-box inference errors keep placeholder records, so selection can fall back to Magi. Set `FORCE['paddle_ocr'] = True` to retry cached errors.

</details>

In [7]:
def crop_box(image, box):
    values = np.asarray(box, dtype=float)
    if values.shape != (4,) or not np.isfinite(values).all():
        return None, None
    height, width = image.shape[:2]
    x1, y1, x2, y2 = values
    clipped = [int(np.clip(np.floor(x1), 0, width)), int(np.clip(np.floor(y1), 0, height)),
               int(np.clip(np.ceil(x2), 0, width)), int(np.clip(np.ceil(y2), 0, height))]
    x1, y1, x2, y2 = clipped
    if x2 <= x1 or y2 <= y1:
        return None, clipped
    return np.ascontiguousarray(image[y1:y2, x1:x2]), clipped


def predict_paddle():
    output = [[] for _ in chapter_pages]
    if not any(result['texts'] for result in magi_raw_results):
        return output
    assert paddle.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0, 'Paddle CUDA unavailable'
    engine = PaddleOCR(**PADDLE_SETTINGS)
    try:
        for page_index, (image, result) in enumerate(zip(chapter_pages, magi_raw_results)):
            for text_index, box in enumerate(result['texts']):
                crop, clipped = crop_box(image, box)
                record = dict(page_index=page_index, text_index=text_index, text_id=f'T{text_index+1}',
                              box=box, crop_box=clipped, text='', lines=[], status='invalid_crop', error=None)
                if crop is not None:
                    try:
                        for prediction in engine.predict(input=crop):
                            texts, scores = prediction['rec_texts'], prediction['rec_scores']
                            assert len(texts) == len(scores), 'Paddle text/score count mismatch'
                            for text, score in zip(texts, scores):
                                record['lines'].append(dict(text=str(text), confidence=float(score)))
                        record['text'] = ' '.join(line['text'] for line in record['lines'])
                        record['status'] = 'text' if record['text'].strip() else 'empty'
                    except Exception as error:
                        record.update(text='', lines=[], status='error', error=str(error))
                        print(f'PG{page_index+1}:T{text_index+1} OCR error: {error}')
                output[page_index].append(record)
    finally:
        del engine
        gc.collect()
        paddle.device.cuda.empty_cache()
    return output


In [8]:
paddle_ocr_results = cached_stage('paddle_ocr',
    dict(magi=stage_ref('magi_raw'), paddle=PADDLE_SETTINGS,
         packages={k: PACKAGES[k] for k in ['paddlepaddle-gpu', 'paddleocr', 'paddlex', 'numpy', 'pillow']}),
    predict_paddle)
print('Paddle records:', sum(len(page) for page in paddle_ocr_results))
print('Cached OCR errors:', sum(r['status'] == 'error' for page in paddle_ocr_results for r in page))


Creating model: ('PP-OCRv5_server_det', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ESHAAN\.paddlex\official_models\PP-OCRv5_server_det`.


RUN   paddle_ocr


Creating model: ('en_PP-OCRv5_mobile_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\ESHAAN\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.


Paddle records: 41
Cached OCR errors: 0


## 5. Select OCR without changing text order

<details>
<summary>How this stage works</summary>

Basic sanity means **valid text**, not correct recognition. No word-specific rules or spelling checks.

Select the only passing output; prefer Paddle when both pass. If only one is nonempty but fails validity (e.g. `...`), retain it and flag review. If both are empty or both nonempty outputs fail, select empty text and flag review. Case-insensitive normalized differences are marked `disagreement`.

Cache both raw OCRs, Paddle confidence/status/errors, and every decision. Selected text becomes `per_page_results[page]['ocr']`; original Magi OCR remains in `magi_ocr`.

</details>

In [9]:
def text_validity(text):
    if text is None:
        return dict(passes=False, reason='empty', text='')
    if not isinstance(text, str):
        return dict(passes=False, reason='not_a_string', text='')
    normalized = ' '.join(text.split())
    if not normalized:
        reason = 'empty'
    elif '\ufffd' in text or any(unicodedata.category(c) in {'Cc', 'Cs'} and c not in '\n\r\t' for c in text):
        reason = 'invalid_characters'
    elif not any(c.isalnum() for c in normalized):
        reason = 'no_letters_or_digits'
    else:
        reason = 'valid_text'
    return dict(passes=reason == 'valid_text', reason=reason, text=normalized)


def select_ocr(paddle_text, magi_text, prefer='paddle'):
    assert prefer in {'paddle', 'magi'}, 'Choose paddle or magi'
    checks = {'paddle': text_validity(paddle_text), 'magi': text_validity(magi_text)}
    passing = [source for source, check in checks.items() if check['passes']]
    nonempty = [source for source, check in checks.items() if check['text']]
    if len(passing) == 1:
        source, reason = passing[0], 'only_valid_output'
    elif len(passing) == 2:
        source, reason = prefer, 'both_valid_preferred'
    elif len(nonempty) == 1:
        source, reason = nonempty[0], 'only_nonempty_needs_review'
    else:
        source = None
        reason = 'both_empty' if all(c['reason'] == 'empty' for c in checks.values()) else 'both_invalid'
    return dict(paddle_text=paddle_text, magi_text=magi_text, validity=checks,
                selected_text=checks[source]['text'] if source else '', selected_source=source,
                reason=reason, needs_review=not bool(passing),
                disagreement=len(passing) == 2 and checks['paddle']['text'].casefold() != checks['magi']['text'].casefold())


def select_sequence_ocr():
    assert len(paddle_ocr_results) == len(magi_raw_results), 'Paddle page count mismatch'
    selected = []
    for page_index, (magi, paddle_records) in enumerate(zip(magi_raw_results, paddle_ocr_results)):
        assert len(paddle_records) == len(magi['texts']) == len(magi['ocr']), 'OCR count mismatch'
        decisions = []
        for text_index, (box, record, magi_text) in enumerate(zip(magi['texts'], paddle_records, magi['ocr'])):
            assert record['page_index'] == page_index and record['text_index'] == text_index, 'OCR order mismatch'
            assert record['box'] == box, 'OCR crop mismatch'
            decision = select_ocr(record['text'], magi_text, PREFERRED_OCR)
            decision.update(page_index=page_index, text_index=text_index, text_id=f'T{text_index+1}', box=box,
                            paddle_lines=record['lines'], paddle_status=record['status'], paddle_error=record['error'])
            decisions.append(decision)
        selected.append(decisions)
    return selected


In [10]:
ocr_selection = cached_stage('ocr_selection',
    dict(magi=stage_ref('magi_raw'), paddle=stage_ref('paddle_ocr'), selector=SELECTOR_SETTINGS),
    select_sequence_ocr)
per_page_results = copy.deepcopy(magi_raw_results)
assert len(ocr_selection) == len(per_page_results), 'Selected OCR page count mismatch'
for page_index, (result, decisions) in enumerate(zip(per_page_results, ocr_selection)):
    assert len(decisions) == len(result['texts']), 'Selected OCR count mismatch'
    for text_index, decision in enumerate(decisions):
        assert decision['page_index'] == page_index and decision['text_index'] == text_index
        assert decision['box'] == result['texts'][text_index], 'Selected OCR box mismatch'
    result['magi_ocr'] = list(result['ocr'])
    result['ocr'] = [record['selected_text'] for record in decisions]
for raw, selected in zip(magi_raw_results, per_page_results):
    assert raw['texts'] == selected['texts'] and len(raw['texts']) == len(selected['ocr'])
display(pd.DataFrame([dict(page=r['page_index']+1, text_id=r['text_id'],
                          paddle=r['paddle_text'], magi=r['magi_text'], selected=r['selected_text'],
                          source=r['selected_source'], reason=r['reason'], review=r['needs_review'],
                          disagreement=r['disagreement']) for page in ocr_selection for r in page]))


RUN   ocr_selection


,page,text_id,paddle,magi,selected,source,reason,review,disagreement
0,1,T1,...THIS IS BAD.,...this is bad.,...THIS IS BAD.,paddle,both_valid_preferred,False,False
1,1,T2,IT LOOKS LIKE THAT PERSON IS GOING TO FOLLOW S...,It looks like that person is going to follow D...,IT LOOKS LIKE THAT PERSON IS GOING TO FOLLOW S...,paddle,both_valid_preferred,False,True
2,1,T3,IF SHE F SUCCEEDS IN LURING OUT ONE PERSON IN ...,If she foods in luring out one person in this ...,IF SHE F SUCCEEDS IN LURING OUT ONE PERSON IN ...,paddle,both_valid_preferred,False,True
3,1,T4,THEN NUT WILL PROBABLY LEAVE.,Then not will probably leave.,THEN NUT WILL PROBABLY LEAVE.,paddle,both_valid_preferred,False,True
4,1,T5,444,...,444,paddle,only_valid_output,False,False
5,1,T6,SASSAN.,Sassan.,SASSAN.,paddle,both_valid_preferred,False,False
6,1,T7,HM?,Hmm?,HM?,paddle,both_valid_preferred,False,True
7,1,T8,"ABOUT SAIKO,","About Saiko,","ABOUT SAIKO,",paddle,both_valid_preferred,False,False
8,1,T9,I GET THE FEELING,I get the feeling,I GET THE FEELING,paddle,both_valid_preferred,False,False
9,1,T10,THAT YOU ARE BEING SOFT ON HER...,that you are being soft on her...,THAT YOU ARE BEING SOFT ON HER...,paddle,both_valid_preferred,False,False


## 6. Remove duplicate character appearances

<details>
<summary>Containment plus embedding similarity, with redirected links</summary>

Keep the outer character box only when it fully contains the inner box, IoU is at least 0.90, and embedding cosine similarity is at least 0.95. These are adjustable heuristic thresholds, not a guarantee of identity. Mere containment does not remove a character.

Compare each inner box directly with an already retained outer box; do not merge through a chain of removed boxes. Surviving characters retain their original relative order. Keep the outer box's embedding, cluster label, and name. Redirect text-character links, deduplicate identical links, and remap character indices. Text boxes, OCR, text-tail links, and reading order stay unchanged.

`character_cleanup.json` stores cleaned pages, aligned embeddings, and the original-to-cleaned index map with IoU/cosine values for every removal. Raw Magi results and raw embeddings stay unchanged. Downstream entities use cleaned character indices; consult the audit to map them back to raw detections. No GT labels are used.

</details>


In [11]:
def remove_duplicate_characters(results, embeddings, min_iou=0.90, min_cosine=0.95):
    """Remove contained, visually similar duplicates; preserve all text indices."""
    assert 0 <= min_iou <= 1 and -1 <= min_cosine <= 1
    assert len(results) == len(embeddings)
    cleaned_pages, cleaned_embeddings, audits = [], [], []
    for page_index, (raw, vectors) in enumerate(zip(results, embeddings)):
        boxes = np.asarray(raw['characters'], dtype=float).reshape(-1, 4)
        assert len(vectors) == len(boxes), 'Character/embedding count mismatch'
        assert len(raw['character_cluster_labels']) == len(boxes)
        areas = np.maximum(0, boxes[:, 2] - boxes[:, 0]) * np.maximum(0, boxes[:, 3] - boxes[:, 1])
        # Largest first. Equal areas prefer the earlier original character index.
        retained, representatives, decisions = [], {}, []
        for inner in sorted(range(len(boxes)), key=lambda i: (-areas[i], i)):
            representative = inner
            for outer in retained:
                a, b = boxes[inner], boxes[outer]
                if not np.isfinite(a).all() or not np.isfinite(b).all() or areas[inner] <= 0 or areas[outer] <= 0:
                    continue
                contained = a[0] >= b[0] and a[1] >= b[1] and a[2] <= b[2] and a[3] <= b[3]
                iou = float(areas[inner] / areas[outer]) if contained else 0.0
                if not contained or iou < min_iou:
                    continue
                x, y = np.asarray(vectors[inner], dtype=float), np.asarray(vectors[outer], dtype=float)
                assert x.ndim == y.ndim == 1 and x.shape == y.shape
                denominator = np.linalg.norm(x) * np.linalg.norm(y)
                if not np.isfinite(x).all() or not np.isfinite(y).all() or not np.isfinite(denominator) or denominator <= 0:
                    continue
                cosine = float(np.clip(np.dot(x, y) / denominator, -1, 1))
                if cosine >= min_cosine:
                    representative = outer
                    decisions.append(dict(removed_original_index=inner, retained_original_index=outer,
                                          smaller_box_coverage=1.0, iou=iou, cosine=cosine))
                    break
            representatives[inner] = representative
            if representative == inner:
                retained.append(inner)
        # Keep the original relative order of surviving character appearances.
        kept = sorted(retained)
        new_index = {old: new for new, old in enumerate(kept)}
        old_to_new = [new_index[representatives[i]] for i in range(len(boxes))]
        page = copy.deepcopy(raw)
        for key in ['characters', 'character_cluster_labels', 'character_names']:
            if key in raw:
                assert len(raw[key]) == len(boxes), f'Misaligned {key}'
                page[key] = [raw[key][i] for i in kept]
        links, seen = [], set()
        for text_index, char_index in raw['text_character_associations']:
            assert 0 <= text_index < len(raw['texts']) and 0 <= char_index < len(boxes)
            link = (text_index, old_to_new[char_index])
            if link not in seen:
                seen.add(link)
                links.append(list(link))
        page['text_character_associations'] = links
        assert page['texts'] == raw['texts'] and page['ocr'] == raw['ocr']
        cleaned_pages.append(page)
        cleaned_embeddings.append([vectors[i] for i in kept])
        audits.append(dict(page_index=page_index, kept_original_indices=kept,
                           original_to_cleaned_index=old_to_new, removals=decisions))
    return dict(pages=cleaned_pages, embeddings=cleaned_embeddings, audit=audits)


character_cleanup = cached_stage('character_cleanup',
    dict(magi=stage_ref('magi_raw'), embeddings=stage_ref('embeddings'),
         cleanup=CHARACTER_CLEANUP_SETTINGS),
    lambda: remove_duplicate_characters(magi_raw_results, embeddings_per_page,
        CHARACTER_CLEANUP_SETTINGS['min_iou'], CHARACTER_CLEANUP_SETTINGS['min_cosine']))
character_results = character_cleanup['pages']
character_embeddings_per_page = character_cleanup['embeddings']
validate_magi(character_results)
# Apply only character fields; preserve the selected OCR already in memory.
for result, cleaned in zip(per_page_results, character_results):
    for key in ['characters', 'character_cluster_labels', 'character_names', 'text_character_associations']:
        if key in cleaned:
            result[key] = copy.deepcopy(cleaned[key])
print('Character counts before:', [len(r['characters']) for r in magi_raw_results])
print('Character counts after: ', [len(r['characters']) for r in character_results])
display(pd.DataFrame([dict(page_index=page['page_index'], **decision)
                      for page in character_cleanup['audit'] for decision in page['removals']]))


CACHE character_cleanup
Character counts before: [11, 10, 46]
Character counts after:  [11, 10, 46]


""


## 7. Cross-page character entities

<details>
<summary>How this stage works</summary>

Adapted from `magiv2_sequence_identity_full.ipynb`, not the annotated viewer's alternate mutual-match algorithm.

Matching unit: Magi page-local character cluster (`PG1:CL1`). Score: maximum member-to-member cosine against an existing entity. Candidates are processed strongest-first. Each local cluster matches at most one entity, and each entity accepts at most one local cluster per page. Unmatched clusters create an `E` ID. Matching candidates can be sorted; text boxes never are.

Cache retains members, character→entity mapping, accepted scores, and strongest evidence pairs. Negative/unassigned cluster labels are treated as separate appearances.

</details>

In [12]:
def page_clusters(result, page_index):
    groups = {}
    for char_index, label in enumerate(result['character_cluster_labels']):
        cluster_id = f'PG{page_index+1}:CL{int(label)+1}' if int(label) >= 0 else f'PG{page_index+1}:C{char_index+1}:unassigned'
        groups.setdefault(cluster_id, []).append(char_index)
    return groups


def clusterize(results, embeddings, threshold=0.50):
    assert -1 <= threshold <= 1, 'Cosine threshold must be between -1 and 1'
    vectors = [np.asarray(page, dtype=float) for page in embeddings]
    assert len(results) == len(vectors)
    for result, page_vectors in zip(results, vectors):
        assert len(result['characters']) == len(page_vectors), 'Embedding count mismatch'
    entities, decisions, char_to_entity = {}, [], {}
    for page_index, result in enumerate(results):
        clusters = page_clusters(result, page_index)
        candidates = []
        for cluster_id, indices in clusters.items():
            for entity_id, entity in entities.items():
                best_score, best_pair = -float('inf'), None
                for current in indices:
                    for member in entity['members']:
                        score = float(np.dot(vectors[page_index][current], vectors[member['page_index']][member['character_index']]))
                        if score > best_score:
                            best_score = score
                            best_pair = dict(current=f'PG{page_index+1}:C{current+1}', existing=member['character_id'])
                if best_score >= threshold:
                    candidates.append(dict(cluster_id=cluster_id, entity_id=entity_id, score=best_score, best_pair=best_pair))
        candidates.sort(key=lambda candidate: candidate['score'], reverse=True)
        accepted, used_entities = {}, set()
        for candidate in candidates:
            if candidate['cluster_id'] not in accepted and candidate['entity_id'] not in used_entities:
                accepted[candidate['cluster_id']] = candidate
                used_entities.add(candidate['entity_id'])
        for cluster_id, indices in clusters.items():
            match = accepted.get(cluster_id)
            entity_id = match['entity_id'] if match else f'E{len(entities)+1}'
            if entity_id not in entities:
                entities[entity_id] = dict(clusters=[], members=[])
            entities[entity_id]['clusters'].append(cluster_id)
            for char_index in indices:
                character_id = f'PG{page_index+1}:C{char_index+1}'
                entities[entity_id]['members'].append(dict(page_index=page_index, character_index=char_index,
                                                         character_id=character_id, cluster_id=cluster_id))
                char_to_entity[character_id] = entity_id
            decisions.append(dict(page_index=page_index, cluster_id=cluster_id, entity_id=entity_id,
                                  reason='matched' if match else 'new_entity',
                                  score=match['score'] if match else None, best_pair=match['best_pair'] if match else None))
    return dict(entities=entities, char_to_entity=char_to_entity, decisions=decisions)


In [13]:
entity_results = cached_stage('entities',
    dict(characters=stage_ref('character_cleanup'), threshold=ENTITY_THRESHOLD,
         algorithm='sequential_best_member_one_cluster_per_entity_per_page'),
    lambda: clusterize(character_results, character_embeddings_per_page, ENTITY_THRESHOLD))
identified_char = entity_results['entities']
char_to_entity = entity_results['char_to_entity']
assert len(char_to_entity) == sum(len(r['characters']) for r in character_results)
print('Entities:', len(identified_char))
display(pd.DataFrame(entity_results['decisions']))


CACHE entities
Entities: 52


,page_index,cluster_id,entity_id,reason,score,best_pair
0,0,PG1:CL1,E1,new_entity,NaN,None
1,0,PG1:CL2,E2,new_entity,NaN,None
2,0,PG1:CL3,E3,new_entity,NaN,None
3,0,PG1:CL4,E4,new_entity,NaN,None
4,0,PG1:CL5,E5,new_entity,NaN,None
5,1,PG2:CL1,E6,new_entity,NaN,None
6,1,PG2:CL2,E7,new_entity,NaN,None
7,1,PG2:CL3,E8,new_entity,NaN,None
8,1,PG2:CL4,E9,new_entity,NaN,None
9,1,PG2:CL5,E10,new_entity,NaN,None


## 8. Link original text indices to entities

<details>
<summary>How this stage works</summary>

Follow `text_index → character_index → E ID`. Keep one final record per original text box, including empty OCR and unassociated text.

A single associated entity populates `entity_id`. No association gives `None`. Multiple associated entities are preserved with an `ambiguous` status instead of silently choosing one. E IDs are scoped by `sequence_id`.

GT labels remain separate. We do not assume GT dialogue order matches detection order or that E1 equals character1.

</details>

In [14]:
def map_text_entities(results, selections, entity_map, sequence_id, page_names):
    assert len(results) == len(selections) == len(page_names)
    records = []
    for page_index, (result, decisions) in enumerate(zip(results, selections)):
        assert len(result['texts']) == len(result['ocr']) == len(decisions)
        links_by_text = {}
        for text_index, char_index in result['text_character_associations']:
            assert 0 <= text_index < len(result['texts']) and 0 <= char_index < len(result['characters'])
            links_by_text.setdefault(text_index, []).append(char_index)
        for text_index, (box, text, decision) in enumerate(zip(result['texts'], result['ocr'], decisions)):
            assert decision['page_index'] == page_index and decision['text_index'] == text_index and decision['box'] == box
            assert decision['selected_text'] == text
            links = []
            for char_index in dict.fromkeys(links_by_text.get(text_index, [])):
                char_id = f'PG{page_index+1}:C{char_index+1}'
                links.append(dict(character_index=char_index, character_id=char_id, entity_id=entity_map[char_id]))
            entity_ids = list(dict.fromkeys(link['entity_id'] for link in links))
            records.append(dict(sequence_id=sequence_id, page_index=page_index, page_file=page_names[page_index],
                                text_index=text_index, text_id=f'T{text_index+1}', box=box, text=text,
                                entity_id=entity_ids[0] if len(entity_ids) == 1 else None,
                                entity_ids=entity_ids, character_links=links,
                                association_status='unassociated' if not links else 'linked' if len(entity_ids) == 1 else 'ambiguous',
                                ocr_source=decision['selected_source'], ocr_reason=decision['reason'],
                                needs_review=decision['needs_review'] or len(entity_ids) > 1,
                                disagreement=decision['disagreement']))
    return records


In [15]:
text_entities = cached_stage('text_entities',
    dict(magi=stage_ref('magi_raw'), characters=stage_ref('character_cleanup'),
         selector=stage_ref('ocr_selection'), entities=stage_ref('entities')),
    lambda: map_text_entities(per_page_results, ocr_selection, char_to_entity, SEQUENCE_ID,
                             [path.name for path in PAGE_FILES]))
assert len(text_entities) == sum(len(r['texts']) for r in magi_raw_results)
columns = ['page_index', 'text_id', 'text', 'entity_id', 'association_status',
           'ocr_source', 'needs_review', 'disagreement']
display(pd.DataFrame(text_entities, columns=columns))


RUN   text_entities


,page_index,text_id,text,entity_id,association_status,ocr_source,needs_review,disagreement
0,0,T1,...THIS IS BAD.,E2,linked,paddle,False,False
1,0,T2,IT LOOKS LIKE THAT PERSON IS GOING TO FOLLOW S...,E2,linked,paddle,False,True
2,0,T3,IF SHE F SUCCEEDS IN LURING OUT ONE PERSON IN ...,E2,linked,paddle,False,True
3,0,T4,THEN NUT WILL PROBABLY LEAVE.,E2,linked,paddle,False,True
4,0,T5,444,E2,linked,paddle,False,False
5,0,T6,SASSAN.,E3,linked,paddle,False,False
6,0,T7,HM?,NaN,unassociated,paddle,False,True
7,0,T8,"ABOUT SAIKO,",E2,linked,paddle,False,False
8,0,T9,I GET THE FEELING,E2,linked,paddle,False,False
9,0,T10,THAT YOU ARE BEING SOFT ON HER...,E2,linked,paddle,False,False


## 9. Selectively view annotated pages
Choose a page and `selected`, `paddle`, or `magi` OCR. Toggle panels, text boxes,
characters, tails, links, OCR, IDs, and entity colors independently. Gray text
boxes indicate nonessential text. Review flags show OCR or association ambiguity.


In [16]:
def center(box):
    x1, y1, x2, y2 = box
    return ((x1+x2)/2, (y1+y2)/2)


def draw_box(ax, box, color, label=None):
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                                   edgecolor=color, facecolor='none'))
    if label:
        ax.text(x1, max(0, y1-4), label, fontsize=8, color='white',
                bbox=dict(facecolor='black', alpha=0.7, edgecolor='none'))


def show_page(page_index=0, panels=False, texts=True, characters=True, tails=False,
              same_character=False, text_character=True, text_tail=False,
              ocr=True, ids=True, entities=True, ocr_source='selected', save_path=None):
    result = per_page_results[page_index]
    assert ocr_source in {'selected', 'paddle', 'magi'}
    cmap = plt.get_cmap('tab20', max(len(identified_char), 1))
    colors = {entity_id: cmap(i) for i, entity_id in enumerate(identified_char)}
    page_records = [r for r in text_entities if r['page_index'] == page_index]
    fig, ax = plt.subplots(figsize=(12, 18))
    ax.imshow(chapter_pages[page_index])
    if panels:
        for i, box in enumerate(result.get('panels', [])):
            draw_box(ax, box, 'green', f'PN{i+1}' if ids else None)
    if texts:
        for i, box in enumerate(result['texts']):
            record = page_records[i]
            essential = result.get('is_essential_text', [])
            color = 'red' if i >= len(essential) or essential[i] else 'gray'
            label = f'T{i+1}' if ids else ''
            if entities:
                label += ' | ' + ('/'.join(record['entity_ids']) or 'unassociated')
            if ocr:
                text = (result['ocr'][i] if ocr_source == 'selected' else
                        ocr_selection[page_index][i][f'{ocr_source}_text']) or ''
                label += ' | ' + (text[:62] + '…' if len(text) > 65 else text)
            if record['needs_review']:
                label += ' [review]'
            draw_box(ax, box, color, label.strip(' |'))
    if characters:
        for i, box in enumerate(result['characters']):
            char_id = f'PG{page_index+1}:C{i+1}'
            entity_id = char_to_entity[char_id]
            label = f'C{i+1} / CL{int(result["character_cluster_labels"][i])+1}' if ids else ''
            if entities:
                label += f' | {entity_id}'
            draw_box(ax, box, colors[entity_id] if entities else 'blue', label.strip(' |'))
    if tails:
        for i, box in enumerate(result.get('tails', [])):
            draw_box(ax, box, 'purple', f'TAIL{i+1}' if ids else None)
    if same_character:
        for indices in page_clusters(result, page_index).values():
            for index in indices[1:]:
                a, b = center(result['characters'][indices[0]]), center(result['characters'][index])
                ax.plot([a[0], b[0]], [a[1], b[1]], color='blue', linewidth=1.5)
    for enabled, key, target, style in [
        (text_character, 'text_character_associations', 'characters', '--'),
        (text_tail, 'text_tail_associations', 'tails', ':')]:
        if enabled:
            for text_index, target_index in result.get(key, []):
                if not (0 <= text_index < len(result['texts']) and 0 <= target_index < len(result.get(target, []))):
                    continue
                a, b = center(result['texts'][text_index]), center(result[target][target_index])
                color = 'purple' if target == 'tails' else 'blue'
                if entities and target == 'characters':
                    color = colors[char_to_entity[f'PG{page_index+1}:C{target_index+1}']]
                ax.plot([a[0], b[0]], [a[1], b[1]], color=color, linestyle=style, linewidth=1.5)
                if ids:
                    target_id = f'C{target_index+1}' if target == 'characters' else f'TAIL{target_index+1}'
                    ax.text((a[0]+b[0])/2, (a[1]+b[1])/2, f'T{text_index+1}→{target_id}', fontsize=7)
    ax.set_title(f'{SEQUENCE_ID} | PG{page_index+1}: {PAGE_FILES[page_index].name} | {ocr_source} OCR')
    ax.axis('off')
    fig.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    plt.close(fig)


def inspect_text(page_index=0, text_index=0):
    decision = ocr_selection[page_index][text_index]
    crop, _ = crop_box(chapter_pages[page_index], decision['box'])
    if crop is not None:
        display(Image.fromarray(crop))
    record = next(r for r in text_entities if r['page_index'] == page_index and r['text_index'] == text_index)
    print(json.dumps(dict(ocr_decision=decision, text_entity=record), ensure_ascii=False, indent=2))


In [17]:
page_control = widgets.Dropdown(options=[(f'PG{i+1} — {p.name}', i) for i, p in enumerate(PAGE_FILES)], description='Page:')
source_control = widgets.Dropdown(options=['selected', 'paddle', 'magi'], description='OCR:')
flags = {name: widgets.Checkbox(value=value, description=name.replace('_', ' '), indent=False)
         for name, value in dict(panels=False, texts=True, characters=True, tails=False,
                                 same_character=False, text_character=True, text_tail=False,
                                 ocr=True, ids=True, entities=True).items()}
controls = dict(page_index=page_control, ocr_source=source_control, **flags)
viewer_output = widgets.interactive_output(show_page, controls)
display(widgets.VBox([widgets.HBox([page_control, source_control]),
                      widgets.HBox(list(flags.values())[:5]), widgets.HBox(list(flags.values())[5:])]), viewer_output)


Output()

In [18]:
# Direct calls, if widgets are unavailable:
# show_page(0, panels=True, tails=True, same_character=True, text_tail=True)
# show_page(0, ocr_source='magi')
# show_page(0, save_path=CACHE_DIR / 'annotations' / '01.png')
# inspect_text(page_index=0, text_index=0)
# for page_index in range(len(chapter_pages)):
#     show_page(page_index)


## 10. Export page dialogue JSON
For each page, keep only text flagged essential by original Magi, preserving
original bubble order. Emit `{"speaker": "E1", "text": "GO."}`. Missing or
ambiguous speaker associations use `null`. Empty essential OCR is retained.

Save `dialogue_pages.json` in this sequence's inference cache. GT matching and
the training-only exclusions `NARRATION`/`UNKNOWN` do not determine this export.


In [19]:
def export_dialogue_pages(results, raw_results, text_records, character_pages):
    assert len(results) == len(raw_results), 'Original page count changed'
    expected_order = [(page_index, text_index)
                      for page_index, result in enumerate(raw_results)
                      for text_index in range(len(result['texts']))]
    actual_order = [(r['page_index'], r['text_index']) for r in text_records]
    assert actual_order == expected_order, 'Original text-entity reading order changed'
    metadata = {(r['page_index'], r['text_index']): r for r in text_records}
    pages, kept_indices, removed_indices = [], [], []

    for page_index, (result, raw) in enumerate(zip(results, raw_results)):
        assert result['texts'] == raw['texts'], f'PG{page_index+1}: original text boxes/order changed'
        assert result['characters'] == character_pages[page_index]['characters'], 'Cleaned character boxes changed'
        assert result['text_character_associations'] == character_pages[page_index]['text_character_associations'], 'Cleaned associations changed'
        assert len(result['ocr']) == len(raw['texts']), 'OCR/text count mismatch'
        essential = raw.get('is_essential_text')
        assert essential is not None and len(essential) == len(raw['texts']), 'Missing or misaligned essential flags'
        assert all(isinstance(flag, bool) for flag in essential), 'Expected boolean essential flags'
        page, kept, removed = [], [], []
        for text_index, text in enumerate(result['ocr']):
            record = metadata[(page_index, text_index)]
            assert record['box'] == raw['texts'][text_index] and record['text'] == text, 'Text-entity alignment changed'
            assert isinstance(text, str), 'Selected OCR must be a string'
            if not essential[text_index]:
                removed.append(text_index)
                continue
            speaker = record['entity_id']
            assert speaker is None or isinstance(speaker, str), 'Entity ID must be a string or null'
            page.append(dict(speaker=speaker, text=text))
            kept.append(text_index)
        assert kept == [index for index, flag in enumerate(essential) if flag], 'Retained reading order changed'
        pages.append(page)
        kept_indices.append(kept)
        removed_indices.append(removed)
    return dict(pages=pages, kept_text_indices=kept_indices, removed_text_indices=removed_indices)


In [20]:
FORCE.setdefault('dialogue_export', False)
STAGE_VERSIONS.setdefault('dialogue_export', 1)

# Check current in-memory alignment even when the export cache exists.
current_dialogue_export = export_dialogue_pages(per_page_results, magi_raw_results, text_entities, character_results)
dialogue_export = cached_stage('dialogue_export',
    dict(magi=stage_ref('magi_raw'), text_entities=stage_ref('text_entities'),
         current_data_sha256=digest(current_dialogue_export),
         filter='magi_is_essential_text', order='original_text_indices', speaker='entity_id_or_null'),
    lambda: current_dialogue_export)
dialogue_pages = dialogue_export['pages']
DIALOGUE_EXPORT_PATH = CACHE_DIR / 'dialogue_pages.json'
write_json(DIALOGUE_EXPORT_PATH, dialogue_pages)
print('Saved:', DIALOGUE_EXPORT_PATH)


CACHE dialogue_export
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleOCR-Magi\cache_inference\test\seq_14469aeb38963085\dialogue_pages.json


In [21]:
export_summary = [dict(page=page_index+1, page_file=PAGE_FILES[page_index].name,
                       original_bubbles=len(magi_raw_results[page_index]['texts']),
                       exported_bubbles=len(page),
                       retained_text_ids=[f'T{i+1}' for i in dialogue_export['kept_text_indices'][page_index]],
                       removed_text_ids=[f'T{i+1}' for i in dialogue_export['removed_text_indices'][page_index]],
                       null_speakers=sum(item['speaker'] is None for item in page))
                  for page_index, page in enumerate(dialogue_pages)]
display(pd.DataFrame(export_summary))
print(json.dumps(dialogue_pages, ensure_ascii=False, indent=2))


,page,page_file,original_bubbles,exported_bubbles,retained_text_ids,removed_text_ids,null_speakers
0,1,01.png,15,15,"[T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11,...",[],1
1,2,02.png,14,14,"[T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11,...",[],2
2,3,03.png,12,9,"[T1, T2, T6, T7, T8, T9, T10, T11, T12]","[T3, T4, T5]",2


[
  [
    {
      "speaker": "E2",
      "text": "...THIS IS BAD."
    },
    {
      "speaker": "E2",
      "text": "IT LOOKS LIKE THAT PERSON IS GOING TO FOLLOW S HER."
    },
    {
      "speaker": "E2",
      "text": "IF SHE F SUCCEEDS IN LURING OUT ONE PERSON IN THIS PLACE,"
    },
    {
      "speaker": "E2",
      "text": "THEN NUT WILL PROBABLY LEAVE."
    },
    {
      "speaker": "E2",
      "text": "444"
    },
    {
      "speaker": "E3",
      "text": "SASSAN."
    },
    {
      "speaker": null,
      "text": "HM?"
    },
    {
      "speaker": "E2",
      "text": "ABOUT SAIKO,"
    },
    {
      "speaker": "E2",
      "text": "I GET THE FEELING"
    },
    {
      "speaker": "E2",
      "text": "THAT YOU ARE BEING SOFT ON HER..."
    },
    {
      "speaker": "E2",
      "text": "YOU GOT SOME REASON FOR THAT?"
    },
    {
      "speaker": "E2",
      "text": "464"
    },
    {
      "speaker": "E2",
      "text": "BEING SOFT..."
    },
    {
      "speaker": "E2",
    

In [22]:
print('Cache stages:')
display(pd.DataFrame([dict(stage=name, file=str(CACHE_DIR / entry['file']),
                          created_at=entry['created_at']) for name, entry in manifest['stages'].items()]))
print('Entities:', len(identified_char), '| original text boxes:', len(text_entities))
print('No labels were loaded. Outputs are inferred identities and OCR, not measured accuracy.')


Cache stages:


,stage,file,created_at
0,magi_raw,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-11T15:08:44.831503+00:00
1,embeddings,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-11T15:08:45.423505+00:00
2,paddle_ocr,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-13T13:20:01.214865+00:00
3,ocr_selection,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-13T13:20:01.252288+00:00
4,character_cleanup,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-11T15:08:53.538463+00:00
5,entities,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-11T15:08:53.706285+00:00
6,text_entities,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-13T13:20:01.443733+00:00
7,narration_visual,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-11T15:48:20.399747+00:00
8,narration_semantic,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-11T15:48:25.929241+00:00
9,narration_predictions,C:\Users\ESHAAN\BYTE\Applied-Task\Magi\paddleO...,2026-10-11T15:48:27.160673+00:00


Entities: 52 | original text boxes: 41
No labels were loaded. Outputs are inferred identities and OCR, not measured accuracy.
